# diagonal-via-strides — ex1: extract diagonal of (N, N) via as_strided

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `diagonal-via-strides`. Running the final beacon cell reports progress against the `Numpy: Diagonal via strides` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Diagonal via strides` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`diagonal-via-strides`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "diagonal-via-strides"
DD_SUBTOPIC = "Numpy: Diagonal via strides"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Diagonal via `as_strided` — quick refresher

Given an `(N, N)` matrix `m` stored row-major (row stride = `N`, col stride = `1`), its main diagonal `[m[0,0], m[1,1], ..., m[N-1, N-1]]` lives at offsets `0, N+1, 2(N+1), ...` in memory.

```
m.as_strided(size=(N,), stride=(N + 1,))
```

**Why `N + 1`.** To walk one step along the diagonal, you advance **one row down** (`+N` elements) and **one column right** (`+1` element). Total stride per diagonal-step = `N + 1`.

**Why it's cheaper than `torch.diagonal`.** `torch.diagonal` does the same thing internally but goes through extra dispatching / checks. `as_strided` is a one-line storage-header rewrite — no copy, no allocation, no dispatch.

**Off-diagonals.** The same trick generalizes: the `k`-th off-diagonal starts at offset `k` (positive `k`) or `k * N` (negative `k`) and uses the same `N + 1` stride. Length shrinks to `N - |k|`.

**Trace.** Sum of the diagonal — `m.as_strided(size=(N,), stride=(N+1,)).sum()` — equals `torch.trace(m)`. Same memory access pattern, no allocation.

**Caveat.** Assumes contiguous row-major storage. If `m` is a transpose or other view, read `m.stride()` first and use `m.stride(0) + m.stride(1)` as the diagonal stride.

### Exercise 1 — extract diagonal of (N, N) via as_strided

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `as_strided(size=(N,), stride=(N+1,))` to extract the main diagonal of a contiguous `(N, N)` tensor as a view (no copy) and verify against `torch.diagonal`.
> Keywords: as_strided, diagonal, view, no-copy
> ```

**KCs targeted:** `diagonal-stride-formula`, `diagonal-as-strided-view`

Implement `ex1_diagonal_via_strides(m)`. Given a 2-D tensor `m` of shape `(N, N)` that is contiguous and row-major, return a 1-D length-`N` view that aliases the main diagonal of `m`. **No copy** — the test confirms with `.data_ptr()`.

**Use `as_strided`.** For a row-major `(N, N)` tensor, the row stride is `N` and the column stride is `1`. To walk one step along the main diagonal you advance one row down (`+N`) AND one column right (`+1`). So the diagonal-step stride is `N + 1`:

```
m.as_strided(size=(N,), stride=(N + 1,))
```

Don't use `torch.diagonal(m)`, `m.diag()`, or fancy indexing (`m[range(N), range(N)]`) — the drill is specifically about the strided-view trick.

**The view must alias `m`.** Writing through the returned tensor mutates the diagonal of `m`. The test verifies this with both `.data_ptr()` equality and an in-place write check.

**Boundary.** Assume `m.is_contiguous()` and `m.dim() == 2` and `m.shape[0] == m.shape[1]`. The test honors those.

In [ ]:
def ex1_diagonal_via_strides(m: Tensor) -> Tensor:
    """Return the main diagonal of a contiguous (N, N) tensor as a strided view."""
    raise NotImplementedError()


def _test_ex1():
    # Hand-checkable 4x4 case.
    m = t.arange(16.0).reshape(4, 4).contiguous()
    # m =
    #   [[ 0,  1,  2,  3],
    #    [ 4,  5,  6,  7],
    #    [ 8,  9, 10, 11],
    #    [12, 13, 14, 15]]
    # Diagonal = [0, 5, 10, 15].
    diag = ex1_diagonal_via_strides(m)
    assert diag.shape == (4,), f'expected (4,), got {tuple(diag.shape)}'
    assert diag.dtype == m.dtype
    expected = t.tensor([0.0, 5.0, 10.0, 15.0])
    assert t.allclose(diag, expected), f'diagonal mismatch: {diag.tolist()} vs {expected.tolist()}'

    # Must be a VIEW — share storage with m.
    assert diag.data_ptr() == m.data_ptr(), 'must be a view (share storage with m)'

    # Write-through aliasing: mutate via diag, verify m's diagonal updated.
    diag[2] = -99.0
    assert m[2, 2].item() == -99.0, 'view must alias — write to diag[2] should change m[2,2]'
    # Restore.
    diag[2] = 10.0

    # Cross-check against torch.diagonal on multiple sizes.
    rng = t.Generator().manual_seed(0)
    for N in [1, 2, 3, 5, 8, 16, 64]:
        mk = t.randn(N, N, generator=rng).contiguous()
        ours = ex1_diagonal_via_strides(mk)
        ref  = t.diagonal(mk)
        assert ours.shape == (N,) == ref.shape
        assert t.allclose(ours, ref, atol=1e-7), f'N={N}: ours != t.diagonal'
        # Trace equivalence: sum(diag) == trace.
        assert abs(ours.sum().item() - t.trace(mk).item()) < 1e-4

    # N=1: trivially the single element.
    m1 = t.tensor([[42.0]])
    d1 = ex1_diagonal_via_strides(m1)
    assert d1.shape == (1,)
    assert d1.item() == 42.0

    # Identity matrix → diagonal of all ones.
    I = t.eye(7)
    assert t.allclose(ex1_diagonal_via_strides(I), t.ones(7))
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_diagonal_via_strides(m: Tensor) -> Tensor:
    N = m.shape[0]
    return m.as_strided(size=(N,), stride=(N + 1,))
```

**Why `N + 1`.** For a row-major `(N, N)` matrix, adjacent elements within a row are `1` apart in memory; adjacent rows are `N` apart. A diagonal step moves down AND right simultaneously — that's `N + 1` elements forward in storage. Walk `N` such steps starting from offset 0 and you've visited `[m[0,0], m[1,1], ..., m[N-1, N-1]]`.

**Why this is more general than `torch.diagonal`.** `torch.diagonal` is a high-level op that constructs the diagonal view internally — same end result, but the trick generalizes. Off-diagonals use the SAME `N + 1` stride with a different starting offset; `as_strided(..., storage_offset=k)` gives the `k`-th super-diagonal. Anti-diagonal uses stride `N - 1` from offset `N - 1`. Once you've internalized the storage math, you can extract ANY contiguous-step pattern.

**Why `.data_ptr()` equality matters.** It proves no copy was made. Strided views are O(1) in time and memory; `torch.diagonal` is also O(1) in modern PyTorch (it returns a view too), but historically some libraries materialized — writing the explicit `as_strided` form makes the no-copy contract obvious.

**Non-contiguous caveat.** If `m` is a transpose or other view, `m.stride()` is not `(N, 1)`. The robust version is `m.as_strided(size=(N,), stride=(m.stride(0) + m.stride(1),))` — read the actual strides and sum them. Out of scope for this drill since we restricted to contiguous inputs.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()